In [38]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [2]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

4.57.6
c:\Users\mickt\miniconda3\Lib\site-packages\transformers\__init__.py


In [2]:
df = pd.read_csv("C:/Users/mickt/Downloads/spam(2).csv", encoding="windows-1252")

In [6]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [7]:
df['v1']

0        ham
1        ham
2       spam
3        ham
4        ham
        ... 
5567    spam
5568     ham
5569     ham
5570     ham
5571     ham
Name: v1, Length: 5572, dtype: object

In [18]:
df['v2']

0       Go until jurong point, crazy.. Available only ...
1                           Ok lar... Joking wif u oni...
2       Free entry in 2 a wkly comp to win FA Cup fina...
3       U dun say so early hor... U c already then say...
4       Nah I don't think he goes to usf, he lives aro...
                              ...                        
5567    This is the 2nd time we have tried 2 contact u...
5568                Will Ì_ b going to esplanade fr home?
5569    Pity, * was in mood for that. So...any other s...
5570    The guy did some bitching but I acted like i'd...
5571                           Rofl. Its true to its name
Name: v2, Length: 5572, dtype: object

In [3]:
# Keep only needed columns
df = df[['v1', 'v2']]

# Rename columns
df.columns = ['label', 'text']

# Convert labels to numeric
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print(df.head())
print(df['label'].value_counts())

   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
label
0    4825
1     747
Name: count, dtype: int64


In [4]:
print(df['label'].value_counts(normalize=True))

label
0    0.865937
1    0.134063
Name: proportion, dtype: float64


In [5]:

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

## TF-IDF Logistic regression

In [39]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

In [43]:
model = LogisticRegression(class_weight="balanced")
model.fit(X_train, train_labels)
val_preds = model.predict(X_val)
print(classification_report(val_labels, val_preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.95      0.93      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



### The initial model showed good overall accuracy but suffered from class imbalance. Since the dataset contains 86% ham and 14% spam, the unweighted loss function disproportionately penalizes errors on the majority class. By introducing balanced class weights, the training process penalizes misclassification of spam messages more heavily, reducing bias toward the majority class and improving recall and F1-score for the minority class.

## Transfer learning test on DistilBert

In [6]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128,
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, train_labels)
val_dataset = SpamDataset(val_encodings, val_labels)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs",
    learning_rate=2e-5,
    eval_strategy="steps",
    eval_steps=100,          # evaluate every 100 steps
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
100,No log,0.045186
200,No log,0.042997
300,No log,0.042389
400,No log,0.046842
500,0.054700,0.042499
600,0.054700,0.039140
700,0.054700,0.039192
800,0.054700,0.038958


c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pi

TrainOutput(global_step=837, training_loss=0.03637525286156074, metrics={'train_runtime': 2041.3971, 'train_samples_per_second': 6.55, 'train_steps_per_second': 0.41, 'total_flos': 442805396857344.0, 'train_loss': 0.03637525286156074, 'epoch': 3.0})

In [10]:
trainer.evaluate()

c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.03895809128880501,
 'eval_runtime': 24.2008,
 'eval_samples_per_second': 46.073,
 'eval_steps_per_second': 2.892,
 'epoch': 3.0}

### I am training the model on 3 epoch at 2e-5 of learning rate with metric Accuracy; F1 score; Precision; Recall. 

In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_dir="./logs",
    learning_rate=2e-5,
    load_best_model_at_end=True
)


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)
trainer.train()

c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
100,No log,0.050390,0.991031,0.965753,0.986014,0.946309
200,No log,0.052795,0.993722,0.976109,0.993056,0.959732
300,No log,0.059572,0.991928,0.969697,0.972973,0.966443
400,No log,0.056110,0.991928,0.969697,0.972973,0.966443
500,0.009200,0.053989,0.992825,0.972789,0.986207,0.959732
600,0.009200,0.053055,0.992825,0.972603,0.993007,0.953020
700,0.009200,0.047584,0.993722,0.976109,0.993056,0.959732
800,0.009200,0.047380,0.993722,0.976109,0.993056,0.959732


c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pi

TrainOutput(global_step=837, training_loss=0.00557187957112792, metrics={'train_runtime': 1942.199, 'train_samples_per_second': 6.884, 'train_steps_per_second': 0.431, 'total_flos': 442805396857344.0, 'train_loss': 0.00557187957112792, 'epoch': 3.0})

### Score

In [15]:
trainer.evaluate()

c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.047379642724990845,
 'eval_accuracy': 0.9937219730941704,
 'eval_f1': 0.9761092150170648,
 'eval_precision': 0.9930555555555556,
 'eval_recall': 0.959731543624161,
 'eval_runtime': 29.6472,
 'eval_samples_per_second': 37.609,
 'eval_steps_per_second': 2.361,
 'epoch': 3.0}

## Transfer learning test on BERT

### Token

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128,
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

### Train/validation split for encodings/label.

In [ ]:
class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, train_labels)
val_dataset = SpamDataset(val_encodings, val_labels)

### Input argument for model.

In [25]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=2e-5,
    load_best_model_at_end=True
)

### metrics collection(Accuracy; F1_score; Précision, Recall)

In [30]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

### Training 

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)
trainer.train()

c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.044285,0.991031,0.966443,0.966443,0.966443
2,0.046700,0.043618,0.992825,0.972789,0.986207,0.959732


c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=558, training_loss=0.04193154586258755, metrics={'train_runtime': 2465.4128, 'train_samples_per_second': 3.616, 'train_steps_per_second': 0.226, 'total_flos': 586342986869760.0, 'train_loss': 0.04193154586258755, 'epoch': 2.0})

In [29]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\mickt\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mickt\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
trainer.evaluate()

c:\Users\mickt\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.04361816495656967,
 'eval_accuracy': 0.9928251121076234,
 'eval_f1': 0.9727891156462585,
 'eval_precision': 0.9862068965517241,
 'eval_recall': 0.959731543624161,
 'eval_runtime': 56.4204,
 'eval_samples_per_second': 19.762,
 'eval_steps_per_second': 1.241,
 'epoch': 2.0}

## Bert has shown equal to slightly lower performance for an increase amount of time per epoch. Bert : 56.42 DistilBert : 29.64